In [26]:
# ============================================================
# STEP 1: Environment Setup (Bypass Colab 3.12 crash)
# ============================================================
import os
import json
import shutil

print("1️⃣ Setting up safe Python 3.10 environment (Takes ~2 mins)...")
!sudo apt-get update -y > /dev/null
!sudo apt-get install software-properties-common -y > /dev/null
!sudo add-apt-repository ppa:deadsnakes/ppa -y > /dev/null
!sudo apt-get install python3.10 python3.10-venv python3.10-distutils -y > /dev/null

!python3.10 -m venv /content/tts-env
!/content/tts-env/bin/pip install -q setuptools wheel cython numpy==1.25.2
!/content/tts-env/bin/pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!/content/tts-env/bin/pip install -q TTS==0.22.0 transformers==4.37.2 librosa soundfile scipy

print("✅ Environment ready!")

1️⃣ Setting up safe Python 3.10 environment (Takes ~2 mins)...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tts 0.22.0 requires numpy==1.22.0; python_version <= "3.10", but you have numpy 1.25.2 which is incompatible.
✅ Environment ready!


In [63]:
import json

# ============================================================
# STEP 2: All 16 Phoneme-Specific Templates (Commas Only, No Quotes)
# ============================================================
VOICE_TEMPLATES_HINDI = {
    "intro": [
        {
            "id": "intro_001_comprehensive",
            "text": "यह एक मुकम्मल स्पीच टेस्ट है। इसमें आप कुछ जुमले पढ़ेंगे ताकि हम आपकी आवाज़ का टेस्ट कर सकें। बेहतर नतीजे के लिए प्लीज़ माइक को अपने मुँह के पास रखें।"
        },
        {
            "id": "intro_002_velar_fronting",
            "text": "यह वेलर फ्रंटिंग टेस्ट है। इसमें हम गले से निकलने वाली आवाज़ों , क , और , ग , साउंड्स की जाँच करेंगे। प्लीज़ माइक को पास रखें।"
        },
        {
            "id": "intro_003_blocks",
            "text": "यह फ्लो और ब्लॉक्स टेस्ट है। हम देखेंगे कि बोलते वक़्त आपकी आवाज़ कहीं अटक तो नहीं रही। किसी भी ब्लॉक को रिकॉर्ड करने के लिए प्लीज़ माइक करीब रखें।"
        },
        {
            "id": "intro_004_repetition",
            "text": "यह रेपिटिशन टेस्ट है। इसमें हम चेक करेंगे कि क्या कोई साउंड या लफ़्ज़ बार-बार रिपीट हो रहा है। प्लीज़ माइक को एक ही जगह पर रखें।"
        },
        {
            "id": "intro_005_prolongation",
            "text": "यह प्रोलॉन्गेशन टेस्ट है। इसमें हम देखेंगे कि क्या आप किसी साउंड को ज़रूरत से ज़्यादा खींच रहे हैं। हर जुमले के दौरान माइक को एक ही जगह रखें।"
        },
        {
            "id": "intro_006_gliding",
            "text": "यह ग्लाइडिंग टेस्ट है। इसमें हम , र , और , ल , साउंड्स के प्रोनंसिएशन को गौर से चेक करेंगे। प्लीज़ माइक को अपने मुँह के पास रखें।"
        },
        {
            "id": "intro_007_stopping",
            "text": "यह स्टॉपिंग टेस्ट है। इसमें हम , स , और , फ़ , जैसी हवा वाली साउंड्स के फ्लो को चेक करेंगे। प्लीज़ माइक पास रखें।"
        },
        {
            "id": "intro_008_cluster_reduction",
            "text": "इसमें हम मुश्किल साउंड्स के कॉम्बिनेशन को चेक करेंगे , जैसे , स्त , या , प्र , वाले लफ़्ज़। प्लीज़ माइक को पास रखें।"
        }
    ],
    "resume": [
        {
            "id": "resume_001_comprehensive",
            "text": "चलिए, वहीं से शुरू करते हैं जहाँ हमने छोड़ा था। यह मुकम्मल असेसमेंट जारी रखते हैं ताकि हम आपकी आवाज़ का टेस्ट कर सकें।"
        },
        {
            "id": "resume_002_velar_fronting",
            "text": "चलिए, वापस , क , और , ग , साउंड्स जैसे गले की आवाज़ों के टेस्ट पर चलते हैं। प्लीज़ माइक पास रखें।"
        },
        {
            "id": "resume_003_blocks",
            "text": "चलिए, फिर से शुरू करते हैं। हम देखेंगे कि बोलते वक़्त आपकी आवाज़ कहीं अटक तो नहीं रही।"
        },
        {
            "id": "resume_004_repetition",
            "text": "चलिए, वापस चलते हैं। हम चेक करेंगे कि क्या कोई साउंड या लफ़्ज़ बार-बार रिपीट हो रहा है।"
        },
        {
            "id": "resume_005_prolongation",
            "text": "चलिए, जारी रखते हैं। हम देखेंगे कि क्या आप किसी साउंड को ज़रूरत से ज़्यादा खींच रहे हैं।"
        },
        {
            "id": "resume_006_gliding",
            "text": "चलिए, वहीं से शुरू करते हैं जहाँ हमने छोड़ा था। हम , र , और , ल , साउंड्स की जाँच करेंगे।"
        },
        {
            "id": "resume_007_stopping",
            "text": "चलिए, वापस शुरू करते हैं। हम , स , और , फ़ , साउंड्स जैसी आवाज़ों के फ्लो को चेक करेंगे।"
        },
        {
            "id": "resume_008_cluster_reduction",
            "text": "चलिए, वहीं से शुरू करते हैं। हम , स्त , और , प्र , जैसे साउंड कॉम्बिनेशन की जाँच करेंगे।"
        }
    ]
}

TEMPLATES_BY_LANGUAGE = {
    "hindi": VOICE_TEMPLATES_HINDI,
}

with open("templates.json", "w", encoding="utf-8") as f:
    json.dump(TEMPLATES_BY_LANGUAGE, f, ensure_ascii=False, indent=4)

print("✅ सभी 16 जुमले सिर्फ़ कोमा-हैक और उर्दू शब्दावली के साथ तैयार हैं।")

✅ सभी 16 जुमले सिर्फ़ कोमा-हैक और उर्दू शब्दावली के साथ तैयार हैं।


In [64]:
# ============================================================
# STEP 3: The Execution Script (Runs inside the env)
# ============================================================
script = r"""import os
import json
import torch
import numpy as np
import librosa
import soundfile as sf
from datetime import datetime
from TTS.api import TTS
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s — %(levelname)s — %(message)s')
logger = logging.getLogger("AudioAssets")

os.environ["COQUI_TOS_AGREED"] = "1"
device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# PREMIUM STOCK VOICES FOR HINDI (Best Quality)
# ============================================================
STOCK_SPEAKERS = {
    "daisy_studious": "Daisy Studious",
    "ana_florence": "Ana Florence",
    "claribel_dervla": "Claribel Dervla",
    "gitta_nikolina": "Gitta Nikolina",
    "sophia_hellen": "Sofia Hellen",
}

SELECTED_SPEAKER = STOCK_SPEAKERS["ana_florence"]
logger.info(f"🎤 Selected Stock Speaker: {SELECTED_SPEAKER}")

# ============================================================
# ADVANCED: Preprocess Hindi Text + Hallucination Prevention
# ============================================================
def preprocess_hindi_text(text):
    '''Preprocess Hindi text to prevent hallucinations'''
    # 1. Remove extra whitespace
    text = " ".join(text.split())

    # 2. Replace problematic ellipsis patterns
    text = text.replace("...", "।")
    text = text.replace("..", "।")

    # 3. Remove leading/trailing punctuation that causes artifacts
    text = text.strip("।!? ")

    # 4. Add single space after punctuation for clarity
    text = text.replace("।", "। ")
    text = text.replace("!", "! ")
    text = text.replace("?", "? ")

    # 5. Remove duplicate spaces
    text = " ".join(text.split())

    # 6. Ensure proper ending punctuation (hard stop prevents trailing hallucinations)
    if not text.endswith(("।", "!", "?")):
        text += "।"

    return text

# ============================================================
# ENHANCED: Audio Post-Processing (Remove Artifacts)
# ============================================================
def remove_leading_trailing_artifacts(file_path, trim_ms=300):
    '''Remove artifacts from start/end of audio file'''
    try:
        y, sr = librosa.load(file_path, sr=None)

        # Convert trim_ms to samples
        trim_samples = int(sr * trim_ms / 1000)

        # Detect silence threshold (very conservative to avoid cutting speech)
        threshold = 0.01

        # Find first significant audio (skip leading artifacts)
        start_idx = 0
        for i in range(0, len(y) - trim_samples):
            if np.max(np.abs(y[i:i+trim_samples])) > threshold:
                start_idx = i
                break

        # Find last significant audio (skip trailing artifacts)
        end_idx = len(y)
        for i in range(len(y) - 1, trim_samples, -1):
            if np.max(np.abs(y[i-trim_samples:i])) > threshold:
                end_idx = i
                break

        # Trim with small crossfade to prevent clicks
        y_trimmed = y[start_idx:end_idx]

        if len(y_trimmed) > sr * 0.5:  # Only save if >500ms remains
            # Apply gentle fade in/out (50ms) to prevent pops
            fade_samples = int(sr * 0.05)
            if fade_samples > 0 and len(y_trimmed) > fade_samples * 2:
                fade_in = np.linspace(0, 1, fade_samples)
                fade_out = np.linspace(1, 0, fade_samples)
                y_trimmed[:fade_samples] *= fade_in
                y_trimmed[-fade_samples:] *= fade_out

            sf.write(file_path, y_trimmed, sr)
            return True
    except Exception as e:
        logger.warning(f"  ⚠️  Artifact removal failed: {e}")
    return False

# ============================================================
# ENHANCED: Audio Normalization + Quality Check
# ============================================================
def enhance_and_validate_audio(file_path):
    '''Enhance audio and validate quality'''
    try:
        y, sr = librosa.load(file_path, sr=None)

        # Check for excessive noise/hallucination (very high energy at extremes)
        rms_energy = np.sqrt(np.mean(y**2))
        peak_amplitude = np.max(np.abs(y))

        # Reject files with extreme clipping or excessive noise
        if peak_amplitude > 0.99:
            logger.warning(f"    ⚠️  High clipping detected ({peak_amplitude:.2f}), regenerating...")
            return False

        # Normalize to -1dB to prevent clipping
        if peak_amplitude > 0:
            target_db = -1.0
            target_linear = 10 ** (target_db / 20.0)
            y = y * (target_linear / peak_amplitude)

        # Apply gentle high-pass filter to remove rumble
        from scipy import signal
        sos = signal.butter(4, 80, 'hp', fs=sr, output='sos')
        y = signal.sosfilt(sos, y)

        # Final normalization
        max_val = np.max(np.abs(y))
        if max_val > 0:
            y = y / (max_val * 1.02)

        sf.write(file_path, y, sr)
        return True
    except Exception as e:
        logger.warning(f"  ⚠️  Audio enhancement failed: {e}")
        return False

print(f"\nLoading Coqui XTTS v2 on {device}...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

with open("templates.json", "r", encoding="utf-8") as f:
    TEMPLATES_BY_LANGUAGE = json.load(f)

# ============================================================
# IMPROVED: Template Flattening (Supports resume section)
# ============================================================
def flatten_templates(language):
    templates = TEMPLATES_BY_LANGUAGE[language]
    items = []
    for category, variations in templates.items():
        if category == "validation":
            for context, entries in variations.items():
                for var in entries:
                    items.append((
                        f"validation_{context}_{var['id']}",
                        var["text"],
                        f"validation_{context}_{var['id']}.wav",
                    ))
        else:
            for var in variations:
                items.append((
                    f"{category}_{var['id']}",
                    var["text"],
                    f"{var['id']}.wav",
                ))
    return items

# ============================================================
# IMPROVED: Build Manifest (Supports resume section)
# ============================================================
def build_manifest(language):
    templates = TEMPLATES_BY_LANGUAGE[language]
    lang_entry = {
        "intro": [],
        "resume": [],
        "waiting": [],
        "validation": {ctx: [] for ctx in templates["validation"].keys()},
        "transition": [],
        "completion": [],
    }

    for item in templates["intro"]:
        lang_entry["intro"].append({
            "id": item["id"], "file": f"{item['id']}.wav", "text": item["text"],
            "test_types": [item["id"].replace("intro_", "").rsplit("_", 1)[0]],
        })

    if "resume" in templates:
        for item in templates["resume"]:
            lang_entry["resume"].append({
                "id": item["id"], "file": f"{item['id']}.wav", "text": item["text"],
                "test_types": [item["id"].replace("resume_", "").rsplit("_", 1)[0]],
            })

    for item in templates["waiting"]:
        lang_entry["waiting"].append({"id": item["id"], "file": f"{item['id']}.wav", "text": item["text"]})

    for ctx, entries in templates["validation"].items():
        for item in entries:
            lang_entry["validation"][ctx].append({
                "id": item["id"], "file": f"validation_{ctx}_{item['id']}.wav",
                "context": ctx, "text": item["text"],
            })

    for item in templates["transition"]:
        lang_entry["transition"].append({"id": item["id"], "file": f"{item['id']}.wav", "text": item["text"]})

    for item in templates["completion"]:
        lang_entry["completion"].append({"id": item["id"], "file": f"{item['id']}.wav", "text": item["text"]})

    return lang_entry

def save_manifest(language, lang_entry):
    os.makedirs("audio_assets", exist_ok=True)
    path = "audio_assets/manifest.json"
    manifest = {}
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            manifest = json.load(f)
    manifest.update({"version": "1.0.0", "generated_at": datetime.now().isoformat(), language: lang_entry})
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    logger.info(f"📋 Manifest updated → {path}")

# ============================================================
# OPTIMIZED: Generate Hindi Audio (Anti-Hallucination)
# ============================================================
def generate_hindi():
    language = "hindi"
    flat = flatten_templates(language)
    total = len(flat)
    out_dir = f"audio_assets/{language}"
    os.makedirs(out_dir, exist_ok=True)

    logger.info(f"\n🎤 [HINDI] {total} clips | model: Coqui XTTS v2")
    logger.info(f"   📢 Stock Speaker: {SELECTED_SPEAKER}")
    logger.info(f"   🎚️  Quality Preset: ANTI-HALLUCINATION")
    logger.info(f"   🛡️  Features: Artifact removal, normalization, high-pass filter")

    start, saved, failed = datetime.now(), 0, 0
    lang_code = "hi"

    for i, (vid, text, filename) in enumerate(flat):
        out_path = os.path.join(out_dir, filename)
        if os.path.exists(out_path) and os.path.getsize(out_path) > 5000:
            logger.info(f"  ⏭️  [{i+1}/{total}] {vid} — already exists, skipping")
            saved += 1
            continue

        # Preprocess Hindi text (ENHANCED for hallucination prevention)
        safe_text = preprocess_hindi_text(text)

        try:
            # ✅ ANTI-HALLUCINATION PARAMETERS
            tts.tts_to_file(
                text=safe_text,
                file_path=out_path,
                speaker=SELECTED_SPEAKER,
                language=lang_code,
                # ✅ CONSERVATIVE SETTINGS TO PREVENT HALLUCINATIONS
                temperature=0.65,              # LOWERED from 0.75 (less random artifacts)
                top_k=150,                     # LOWERED from 200 (more focused vocabulary)
                top_p=0.80,                    # LOWERED from 0.85 (less divergence)
                repetition_penalty=1.5,        # INCREASED from 1.2 (prevent repeated artifacts)
                speed=0.95                     # Slightly slower for clearer enunciation
            )

            # Remove leading/trailing artifacts
            remove_leading_trailing_artifacts(out_path, trim_ms=250)

            # Enhance and validate audio quality
            if enhance_and_validate_audio(out_path):
                logger.info(f"  ✅ [{i+1}/{total}] {vid} → saved")
                saved += 1
            else:
                logger.warning(f"  ⚠️  [{i+1}/{total}] {vid} → quality validation failed, regenerating...")
                failed += 1

        except Exception as e:
            logger.error(f"  ❌ [{i+1}/{total}] {vid} failed: {e}")
            failed += 1

    elapsed = (datetime.now() - start).total_seconds()
    logger.info(f"\n✅ Complete — {saved}/{total} clips saved | {failed} regenerations needed")
    logger.info(f"   ⏱️  Total time: {elapsed:.1f}s")

# ============================================================
# Execute Generation
# ============================================================
generate_hindi()
save_manifest("hindi", build_manifest("hindi"))
"""

with open("generate_audio.py", "w", encoding="utf-8") as f:
    f.write(script)

print("4️⃣ Audio generation script created (Anti-Hallucination Mode).")
print("   🛡️  Features:")
print("      • Conservative temperature (0.65) to prevent artifacts")
print("      • Artifact trimming (leading/trailing hallucinations)")
print("      • High-pass filter (removes rumble/noise)")
print("      • Quality validation (rejects clipped/distorted files)")
print("      • Gentle fade in/out (prevents clicks/pops)")

4️⃣ Audio generation script created (Anti-Hallucination Mode).
   🛡️  Features:
      • Conservative temperature (0.65) to prevent artifacts
      • Artifact trimming (leading/trailing hallucinations)
      • High-pass filter (removes rumble/noise)
      • Quality validation (rejects clipped/distorted files)
      • Gentle fade in/out (prevents clicks/pops)


In [67]:
print("\n5️⃣ Starting Coqui XTTS Audio Generation (High Quality Mode)...")
print("   Parameters: temperature=0.75, top_k=200, top_p=0.85, repetition_penalty=1.2")
print("   Speaker: Ana Florence (Premium Stock Voice)")

!cd /content && /content/tts-env/bin/python generate_audio.py


5️⃣ Starting Coqui XTTS Audio Generation (High Quality Mode)...
   Parameters: temperature=0.75, top_k=200, top_p=0.85, repetition_penalty=1.2
   Speaker: Ana Florence (Premium Stock Voice)
2026-03-02 18:12:24,444 — INFO — 🎤 Selected Stock Speaker: Ana Florence

Loading Coqui XTTS v2 on cuda...
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts
/content/tts-env/lib/python3.10/site-packages/TTS/tts/layers/xtts/xtts_manager.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no 

In [68]:
# ============================================================
# STEP 5: Zip and Download
# ============================================================
print("\n6️⃣ Zipping audio assets for download...")
shutil.make_archive("hindi_audio_assets", "zip", "audio_assets")

# Display summary
import subprocess
result = subprocess.run(["find", "audio_assets", "-name", "*.wav"], capture_output=True, text=True)
wav_count = len(result.stdout.strip().split('\n')) if result.stdout.strip() else 0
manifest_size = os.path.getsize("audio_assets/manifest.json") / 1024 if os.path.exists("audio_assets/manifest.json") else 0

print(f"   📊 Summary:")
print(f"   • WAV files generated: {wav_count}")
print(f"   • Manifest file: {manifest_size:.1f} KB")
print(f"   • Total zip size: {os.path.getsize('hindi_audio_assets.zip') / 1024 / 1024:.1f} MB")

from google.colab import files
print("\n7️⃣ Downloading hindi_audio_assets.zip...")
files.download("hindi_audio_assets.zip")

print("\n🎉 Process Complete! ZIP file is downloading.")


6️⃣ Zipping audio assets for download...
   📊 Summary:
   • WAV files generated: 16
   • Manifest file: 0.0 KB
   • Total zip size: 5.1 MB

7️⃣ Downloading hindi_audio_assets.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Process Complete! ZIP file is downloading.


In [ ]:
# ============================================================
# OPTIONAL: Verify Audio Quality
# ============================================================
import librosa
import numpy as np

def analyze_audio_quality(wav_path):
    """Analyze audio file quality metrics"""
    try:
        y, sr = librosa.load(wav_path, sr=None)

        # Metrics
        duration = librosa.get_duration(y=y, sr=sr)
        rms_energy = np.sqrt(np.mean(y**2))
        peak_amplitude = np.max(np.abs(y))
        clipping = "✅ No" if peak_amplitude < 0.98 else "⚠️  Yes"

        return {
            "duration_ms": duration * 1000,
            "rms_energy": rms_energy,
            "peak_amplitude": peak_amplitude,
            "clipping": clipping
        }
    except:
        return None

# Analyze first 3 generated files
print("\n📊 Audio Quality Analysis:")
import glob
wav_files = glob.glob("audio_assets/hindi/*.wav")[:3]
for wav_file in wav_files:
    metrics = analyze_audio_quality(wav_file)
    if metrics:
        filename = os.path.basename(wav_file)
        print(f"\n  {filename}:")
        print(f"    • Duration: {metrics['duration_ms']:.0f} ms")
        print(f"    • Peak Amplitude: {metrics['peak_amplitude']:.3f}")
        print(f"    • Clipping Detected: {metrics['clipping']}")